# 🎯 [MNT10_M06] 야전 정비 요청 현황 및 이동정비반 관제 대시보드 UI 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT10/mnt10_m06_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT10_M06 |
| **미션 주제** | 야전 정비 요청 현황 및 이동정비반 관제 대시보드 UI 구현 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
야전에서 장비가 고장 나면 이동정비반이 얼마나 빨리 도착해 고치느냐가 부대 전투력 회복 속도를 좌우하지만, 가까운 정비반부터 보내는 방식은 요청이 몰리는 지역의 정비반에 부하를 집중시키고 핵심 장비를 뒤로 미루는 문제가 있다. 본 프로젝트는 정비인력의 부하율과 공정 균형을 데이터로 예측·관리하는 것을 목표로, 교육용 가상 지역의 긴급 정비 요청과 이동정비반 자료를 코드로 만들어 분석한다. 요청 데이터 스키마 정의, 정비반 상태 자료 정제, 거리 기준 Baseline과 탐색적 분석, 중요도·정비 소요시간·이동 경로를 종합한 우선순위 엔진, 출동 지연 위험도와 현장 수리 가능성 점수, 관제 대시보드와 설명가능성, 정비대대장 승인(HITL) 시뮬레이터, 동시 다발 요청 스트레스 시험, AAR 가중치 재튜닝과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 핵심 장비의 가동중단 시간을 줄이면서 정비반 간 부하를 고르게 하고, 출동 순서 변경과 재배치는 지휘관이 근거를 보고 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 산출물(deliverables) 디렉토리 구성 안내

본 과제는 실무 현장에서 바로 쓰일 수 있는 국방 소프트웨어 아키텍처 표준을 지향합니다.  
과제를 진행하며 생성되는 산출물은 아래 규칙에 따라 각 폴더에 체계적으로 저장해 주세요.

| 디렉토리 | 군수 실무 비유 | 주 역할 및 산출물 파일 | 분리 목적 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | 야전 교범 / 정비 지침서 | `mnt10_logistics_domain_spec.md` (미션1), `mnt10_operations_guide.md` (미션10) | 코드를 모르는 지휘관/후임자도 데이터 규격과 비상 매뉴얼을 이해하도록 보장 |
| **`notebooks/`** | 정비 시험실 / 연구실 | `mnt10_eda_analysis.ipynb` (미션3), 본 실습 노트북 | AI와 대화하며 자유롭게 데이터를 시각화하고 알고리즘을 반복 실험 |
| **`src/`** | 실제 장착 핵심 부품 모듈 | `src/etl/`, `src/features/`, `src/models/`, `src/dashboard/`, `src/optimization/` | 운영 서버에서 자동 배치 실행 가능한 순수 파이썬 모듈(`.py`)로 부품화 |
| **`reports/`** | 지휘관 결심 보고서 | `mnt10_roi_analysis.md` (미션9), AAR 오배정 분석서 | 정비대대장/군수참모에게 정비 지연 15% 감축 등 정량 ROI 성과 보고 |
| **`pipeline/`** | 자동화 컨베이어 벨트 | `pipeline/mnt10_retrain.py` (미션10) | 긴급 요청 유입 시 전처리부터 관제까지 자동 구동 및 재학습 파이프라인 |

> **💡 [바이브 코딩(AI 협업) 실습 요령]**  
> 1. **노트북에서 먼저 검증**: 본 노트북의 각 미션 셀에서 AI에게 질의하며 단계별 코드를 실행하고 결과를 확인합니다.  
> 2. **AI에게 모듈화 요청**: 검증 완료 후 AI에게 *"이 코드를 `src/.../` 파일에 넣을 수 있게 함수/클래스로 모듈화해줘"*라고 요청합니다.  
> 3. **디렉토리별 저장**: 제안된 코드를 해당 파일에 저장하여 자연스럽게 실무 패키지를 완성합니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 야전 정비 요청 현황 및 이동정비반 관제 대시보드 UI 구현
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

야전 정비 요청 현황 및 이동정비반 관제 대시보드 UI를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **정비 요청 현황과 이동정비반 상태를 보여 주는 관제 대시보드를 만들고, 예측마다 근거를 설명하는 설명가능성(XAI) 기능을 붙인다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **요청·정비반 지도, 정비반 부하율, 중요도별 대기 요청, 일별·중요도별 도착시간, 예측 워터폴 그림으로 관제 화면을 구성 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/explainability/mnt10_xai_explainer.py 및 XAI 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 장교/군무원이 신뢰하고 조치할 수 있는 설명의 명확성

#### 미션 PASS 조건

- 정량 기준: 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%


- 진행 기준: SHAP summary plot 및 waterfall plot 생성 파이프라인 완성


- 검증 방법: 실무자 대상 설명 근거 유용성 평가 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""야전 정비 요청 현황과 이동정비반 상태(위치·부하율·대기 요청·도착시간 추이)를 한 화면에 보여 주는 관제 대시보드를 그리고, 정비 소요시간 예측마다 기여도 상위 3개 인자를 자동 산출해 워터폴 그림과 자연어 설명으로 제공하는 설명가능성(XAI) 모듈을 구현한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 이동정비반·긴급 정비 요청 생성기 (실제 부대·위치·장비·인원 정보 없음)
# 60km x 60km 가상 지역에 이동정비반 6개와 30일간의 긴급 정비 요청을 만든다.
# 장비 구분(mobility/fires/comms/power)과 고장 유형은 교육용 추상 분류다.
# ---------------------------------------------------------------------------
AREA_KM, DUTY_H = 60, 10.0                     # 가상 작전지역 크기, 정비반 1일 기준 가용시간
FAULT_H = {"engine": 6.0, "drivetrain": 5.0, "hydraulic": 3.0, "electric": 2.0, "comm_module": 1.5, "cooling": 2.5}
FIELD_P = {"engine": 0.3, "drivetrain": 0.35, "hydraulic": 0.75, "electric": 0.9, "comm_module": 0.95, "cooling": 0.8}
EQUIP_F = {"mobility": 1.0, "fires": 1.2, "comms": 0.7, "power": 0.8}
FAULT_OF = {"mobility": [0.3, 0.25, 0.2, 0.1, 0.0, 0.15], "fires": [0.2, 0.15, 0.35, 0.15, 0.0, 0.15],
            "comms": [0.0, 0.0, 0.0, 0.4, 0.5, 0.1], "power": [0.35, 0.0, 0.0, 0.35, 0.0, 0.3]}
CRIT_P = {"mobility": [0.45, 0.35, 0.2], "fires": [0.2, 0.35, 0.45], "comms": [0.3, 0.4, 0.3], "power": [0.6, 0.3, 0.1]}
HUBS = [(15, 20), (45, 25), (30, 50)]           # 가상 훈련장 3곳(요청이 몰리는 지점)


def make_teams(seed=0):
    rng = np.random.default_rng(seed)
    base = [(8, 10), (30, 6), (52, 12), (10, 44), (32, 38), (54, 50)]
    skills = [["fires", "mobility"], ["comms", "mobility"], ["fires", "power"],
              ["mobility", "power"], ["comms", "fires"], ["comms", "mobility", "power"]]
    return [{"team": f"MT{i + 1}", "x": float(x), "y": float(y), "crew": int(rng.integers(3, 6)), "skills": s}
            for i, ((x, y), s) in enumerate(zip(base, skills))]


def make_requests(n_days=30, per_day=10, seed=1, surge_days=(8, 9, 10, 11, 22, 23, 24)):
    """하루 평균 per_day건, 훈련 주간(surge_days)에는 1.6배의 긴급 정비 요청을 만든다."""
    rng = np.random.default_rng(seed)
    rows = []
    for d in range(n_days):
        n = rng.poisson(per_day * (1.6 if d in surge_days else 1.0))
        weather = rng.choice(["clear", "rain", "snow"], p=[0.65, 0.25, 0.1])
        for _ in range(n):
            if rng.random() < 0.7:
                hx, hy = HUBS[rng.integers(0, 3)]
                x, y = np.clip(rng.normal(hx, 6), 0, AREA_KM), np.clip(rng.normal(hy, 6), 0, AREA_KM)
            else:
                x, y = rng.uniform(0, AREA_KM, 2)
            eq = str(rng.choice(list(EQUIP_F), p=[0.45, 0.25, 0.2, 0.1]))
            fault = str(rng.choice(list(FAULT_H), p=FAULT_OF[eq]))
            parts = bool(rng.random() < 0.75)
            bad = weather != "clear"
            std_h = FAULT_H[fault] * EQUIP_F[eq] * (1.35 if not parts else 1.0) * (1.15 if bad else 1.0) * rng.lognormal(0, 0.12)
            rows.append({"t_req": d * 24 + float(np.clip(rng.normal(13, 4), 5, 23)), "day": d,
                         "x": float(x), "y": float(y), "equip": eq, "fault": fault,
                         "crit": int(rng.choice([1, 2, 3], p=CRIT_P[eq])), "parts_ready": parts,
                         "weather": str(weather), "road": "unpaved" if y > 30 and rng.random() < 0.7 else "paved",
                         "field_ok": bool(rng.random() < FIELD_P[fault] * (1.0 if parts else 0.5)),
                         "std_repair_h": round(float(std_h), 2), "road_delay": float(rng.lognormal(0, 0.15))})
    df = pd.DataFrame(rows).sort_values("t_req").reset_index(drop=True)
    df.insert(0, "req_id", [f"R{i:04d}" for i in range(len(df))])
    return df
# ---------------------------------------------------------------------------

# ---- 출동 배정·정비 모의 공통 함수 ----
W0 = {"crit": 1.0, "wait": 0.3, "short": 0.2, "balance": 0.5}   # 우선순위·배정 기본 가중치(미션 9에서 재튜닝)


def travel_h(team, r, actual=False):
    """도로 굴곡 1.3배, 포장 35km/h·비포장 25km/h, 악천후 20% 감속. actual=True면 실제 도로 지연을 반영."""
    d = 1.3 * np.hypot(team["x"] - r["x"], team["y"] - r["y"])
    v = (35.0 if r["road"] == "paved" else 25.0) / (1.2 if r["weather"] != "clear" else 1.0)
    return d / v * (r["road_delay"] if actual else 1.0)


def on_site_h(team, r):
    """현장 정비시간: 4인 기준 표준시간을 투입 인원으로 환산. 현장 수리 불가면 진단·후송 조치 1시간."""
    return r["std_repair_h"] * 4 / team["crew"] if r["field_ok"] else 1.0


def priority(r, now, est_h, w):
    return w["crit"] * r["crit"] + w["wait"] * (now - r["t_req"]) + w["short"] / (1 + est_h)


def simulate(req, teams, policy="baseline", w=None, cycle_h=1.0, est=None):
    """배정 주기(cycle_h)마다 대기 요청을 정렬해 이동정비반에 배정하고 출동·정비·복귀를 모의한다.
    baseline: 접수 순서 + 거리 최근접 정비반 / priority: 우선순위 점수 순서 + 도착 예상시각·당일 부하 균형"""
    w = w or W0
    est = np.asarray(est if est is not None else req["std_repair_h"].values, float)
    n_days = int(req["day"].max()) + 1
    free = {t["team"]: 0.0 for t in teams}
    load = {t["team"]: np.zeros(n_days + 5) for t in teams}
    recs, out, pending, i, now = req.to_dict("records"), [], [], 0, cycle_h
    while i < len(recs) or pending:
        while i < len(recs) and recs[i]["t_req"] <= now:
            pending.append((i, recs[i])); i += 1
        if policy == "priority":
            pending.sort(key=lambda p: -priority(p[1], now, est[p[0]], w))
        for k, r in pending:
            cand = [t for t in teams if r["equip"] in t["skills"]]
            day = min(int(now // 24), n_days + 4)
            if policy == "baseline":
                tm = min(cand, key=lambda t: np.hypot(t["x"] - r["x"], t["y"] - r["y"]))
            else:
                tm = min(cand, key=lambda t: max(free[t["team"]], now) + travel_h(t, r)
                         + w["balance"] * load[t["team"]][day] / DUTY_H * 4)
            backlog = max(0.0, free[tm["team"]] - now)
            go = max(free[tm["team"]], now)
            tr = travel_h(tm, r, actual=True)
            work = on_site_h(tm, r)
            free[tm["team"]] = go + 2 * tr + work
            load[tm["team"]][min(int(go // 24), n_days + 4)] += 2 * tr + work
            out.append({**r, "idx": k, "team": tm["team"], "dist_km": float(np.hypot(tm["x"] - r["x"], tm["y"] - r["y"])),
                        "backlog_h": backlog, "cycle_wait_h": now - r["t_req"], "dispatch": go, "travel_h": tr,
                        "work_h": work, "response_h": go + tr - r["t_req"], "downtime_h": go + tr + work - r["t_req"]})
        pending, now = [], now + cycle_h
    L = pd.DataFrame({k: v[:n_days] for k, v in load.items()}) / DUTY_H      # 정비반별 일일 부하율
    return pd.DataFrame(out).sort_values("idx").reset_index(drop=True), L


def metrics(res, L):
    c3 = res[res["crit"] == 3]
    return {"평균 도착(h)": round(res["response_h"].mean(), 2), "핵심장비 도착(h)": round(c3["response_h"].mean(), 2),
            "핵심장비 90%(h)": round(c3["response_h"].quantile(0.9), 2), "평균 부하율": round(float(L.values.mean()), 3),
            "부하율 편차": round(float(L.mean().std()), 3), "과부하 일수": int((L > 1.0).values.sum()),
            "총 이동(h)": round(float(res["travel_h"].sum() * 2), 1)}

# ---- 접수 시점 파생 피처(미션 4와 동일) ----
def make_features(req, teams):
    """접수 시점에 알 수 있는 정보만으로 파생 피처를 만든다(미래 정보 누설 방지)."""
    X = pd.DataFrame(index=req.index)
    X["hour"] = req["t_req"] % 24
    X["is_night"] = (X["hour"] >= 19).astype(int)
    X["dow"] = req["day"] % 7
    X["weather_bad"] = (req["weather"] != "clear").astype(int)
    X["unpaved"] = (req["road"] == "unpaved").astype(int)
    X["no_parts"] = (~req["parts_ready"]).astype(int)
    X["std_table_h"] = req["fault"].map(FAULT_H) * req["equip"].map(EQUIP_F)        # 정비 표준시간표 값
    X["crit_x_noparts"] = req["crit"] * X["no_parts"]
    X["bad_x_table"] = X["weather_bad"] * X["std_table_h"]
    X["near_team_km"] = [min(np.hypot(t["x"] - x, t["y"] - y) for t in teams if e in t["skills"]) for x, y, e in zip(req["x"], req["y"], req["equip"])]
    X["hub_km"] = [min(np.hypot(hx - x, hy - y) for hx, hy in HUBS) for x, y in zip(req["x"], req["y"])]
    X["req_24h"] = [((req["t_req"] > t - 24) & (req["t_req"] <= t)).sum() for t in req["t_req"]]
    X["fault_past_mean"] = req.groupby("fault")["std_repair_h"].transform(lambda s: s.shift().expanding().mean()).fillna(req["std_repair_h"].mean())
    X["crit"] = req["crit"]
    for e in EQUIP_F:
        X[f"eq_{e}"] = (req["equip"] == e).astype(int)
    return X


def fit_repair_model(hist, teams):
    """과거 요청으로 정비 소요시간(4인 기준) 예측 모델을 학습한다."""
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(make_features(hist, teams), hist["std_repair_h"])

KO = {"std_table_h": "정비 표준시간표 값", "no_parts": "수리부속 미확보", "weather_bad": "악천후", "bad_x_table": "악천후 x 표준시간",
      "crit_x_noparts": "핵심장비 x 부속 미확보", "fault_past_mean": "같은 고장 과거 평균", "unpaved": "비포장 도로", "crit": "작전 중요도",
      "near_team_km": "최근접 정비반 거리", "hub_km": "훈련장과의 거리", "req_24h": "최근 24시간 요청 수", "hour": "접수 시각",
      "is_night": "야간 접수", "dow": "요일"}


def explain(model, X, bg, top=3):
    """가림(occlusion) 방식 국소 기여도: 피처 j를 배경 표본 값으로 바꿨을 때 예측이 평균적으로 얼마나 달라지는지.
    SHAP과 같은 '이 예측을 평균에서 얼마나 밀어 올렸나'를 근사한다(외부 라이브러리 없이)."""
    base = model.predict(X)
    contrib = np.zeros(X.shape)
    for j, c in enumerate(X.columns):
        for b in bg:
            Xb = X.copy(); Xb[c] = b[j]
            contrib[:, j] += base - model.predict(Xb)
    contrib /= len(bg)
    C = pd.DataFrame(contrib, columns=X.columns, index=X.index)
    tops = [list(C.loc[i].abs().sort_values(ascending=False).index[:top]) for i in C.index]
    return base, C, tops


def sentence(r, pred, C, top):
    parts = [f"{KO.get(f, f)}({'+' if C[f] > 0 else ''}{C[f]:.1f}h)" for f in top]
    return f"{r['req_id']} {r['equip']}/{r['fault']} 예상 정비 {pred:.1f}h(평균 대비 {pred - AVG:+.1f}h), 영향 요인: " + ", ".join(parts)


teams = make_teams()
hist, req = make_requests(seed=11), make_requests(seed=1)
model = fit_repair_model(hist, teams)
AVG = float(hist["std_repair_h"].mean())
Xh, X = make_features(hist, teams), make_features(req, teams)
res, L = simulate(req, teams, "priority", est=model.predict(X))

# 1) 설명가능성: 평가 기간 전체 요청에 대해 상위 3개 인자 자동 산출
t0 = time.perf_counter()
bg = Xh.sample(30, random_state=0).values
pred, C, tops = explain(model, X, bg)
sec = time.perf_counter() - t0
rate = np.mean([len(t) == 3 for t in tops])
print(f"[교육용가상자료] 요청 {len(X)}건 설명 산출 {sec:.1f}초, 상위 3개 인자 자동 산출률 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'})")
check = np.abs(C.sum(axis=1) - (pred - model.predict(Xh).mean())).mean()
print(f"기여도 합과 (예측 - 평균 예측)의 평균 차이 {check:.2f}h -> 근사 설명이므로 참고 지표로 쓴다")
freq = pd.Series([f for t in tops for f in t]).map(lambda f: KO.get(f, f)).value_counts().head(5)
print("상위 3개 인자에 자주 등장한 요인:", ", ".join(f"{k} {v}건" for k, v in freq.items()))

# 2) 관제 대시보드: 22일차 14시 기준
day, now = 22, 22 * 24 + 14
cur = res[(res["t_req"] <= now) & (res["t_req"] > now - 24)]
waiting = cur[cur["dispatch"] > now]
print(f"\n[관제 현황 {day}일차 14시] 최근 24시간 요청 {len(cur)}건, 출동 대기 {len(waiting)}건, 정비반 {len(teams)}개")
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
col = {1: "tab:gray", 2: "tab:orange", 3: "tab:red"}
ax[0, 0].scatter(cur["x"], cur["y"], c=cur["crit"].map(col), s=40)
for t in teams:
    ax[0, 0].plot(t["x"], t["y"], "ks"); ax[0, 0].annotate(t["team"], (t["x"] + 1, t["y"] + 1))
ax[0, 0].set_xlim(0, AREA_KM); ax[0, 0].set_ylim(0, AREA_KM); ax[0, 0].set_title("Requests (24h, red=critical) & teams")
lr = L.iloc[day]
ax[0, 1].bar(lr.index, lr.values, color=["tab:red" if v > 1 else "tab:blue" for v in lr.values])
ax[0, 1].axhline(1.0, ls="--", color="k"); ax[0, 1].set_title(f"Team load rate, day {day}")
q = waiting.groupby("crit").size().reindex([1, 2, 3], fill_value=0)
ax[0, 2].bar(["normal", "important", "critical"], q.values, color=[col[k] for k in q.index]); ax[0, 2].set_title("Waiting requests by criticality")
dd = res.groupby("day")["response_h"].mean()
ax[1, 0].plot(dd.index, dd.values, marker="o"); ax[1, 0].set_title("Daily mean response time (h)"); ax[1, 0].set_xlabel("day")
ax[1, 1].boxplot([res.loc[res["crit"] == k, "response_h"] for k in (1, 2, 3)]); ax[1, 1].set_xticks([1, 2, 3], ["normal", "important", "critical"])
ax[1, 1].set_title("Response time by criticality")
i = int(C.index[req["crit"].eq(3)][int(np.argmax(pred[req["crit"].eq(3).values]))])
ci = C.loc[i].reindex(C.loc[i].abs().sort_values(ascending=False).index)[:6][::-1]
start = model.predict(Xh).mean()
ax[1, 2].barh(range(len(ci)), ci.values, left=start + np.r_[0, np.cumsum(ci.values)[:-1]], color=["tab:red" if v > 0 else "tab:green" for v in ci.values])
ax[1, 2].set_yticks(range(len(ci)), ci.index); ax[1, 2].set_title(f"Waterfall: {req.loc[i, 'req_id']} repair time"); ax[1, 2].set_xlabel("hours")
plt.tight_layout(); plt.show()
print("대시보드 구성요소 6종: 요청·정비반 지도, 정비반 부하율, 중요도별 대기, 일별 도착시간, 중요도별 도착시간, 예측 워터폴")

# 3) 자연어 설명 예시(정비대대 관제 화면 하단)
print("\n[예측 설명 예시]")
for k in list(waiting.sort_values("crit", ascending=False)["idx"][:4]) + [i]:
    print(" - " + sentence(req.loc[k], pred[k], C.loc[k], tops[k]))
over = lr[lr > 1.0]
act = "전 정비반 과부하 -> 상급 정비부대 지원 요청·비핵심 요청 연기 검토" if len(over) == len(teams) else "여유 정비반 지원 검토"
print(f"\n경보: {day}일차(훈련 주간) 부하율 1.0 초과 정비반 {list(over.index)} -> {act}(정비대대장 판단)" if len(over) else "\n경보: 과부하 정비반 없음")


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 요청 356건 설명 산출 1.7초, 상위 3개 인자 자동 산출률 100% (목표 100%: 달성)
기여도 합과 (예측 - 평균 예측)의 평균 차이 0.31h -> 근사 설명이므로 참고 지표로 쓴다
상위 3개 인자에 자주 등장한 요인: 정비 표준시간표 값 329건, 핵심장비 x 부속 미확보 227건, 같은 고장 과거 평균 225건, 악천후 x 표준시간 189건, 수리부속 미확보 57건

[관제 현황 22일차 14시] 최근 24시간 요청 22건, 출동 대기 6건, 정비반 6개
대시보드 구성요소 6종: 요청·정비반 지도, 정비반 부하율, 중요도별 대기, 일별 도착시간, 중요도별 도착시간, 예측 워터폴

[예측 설명 예시]
 - R0256 fires/electric 예상 정비 2.4h(평균 대비 -1.7h), 영향 요인: 정비 표준시간표 값(-1.0h), 같은 고장 과거 평균(-0.3h), 악천후 x 표준시간(-0.2h)
 - R0260 mobility/hydraulic 예상 정비 3.3h(평균 대비 -0.8h), 영향 요인: 정비 표준시간표 값(-0.3h), 악천후 x 표준시간(-0.2h), 핵심장비 x 부속 미확보(-0.2h)
 - R0262 power/cooling 예상 정비 2.1h(평균 대비 -2.0h), 영향 요인: 정비 표준시간표 값(-1.3h), 악천후 x 표준시간(-0.2h), 같은 고장 과거 평균(-0.2h)
 - R0261 comms/electric 예상 정비 1.4h(평균 대비 -2.7h), 영향 요인: 정비 표준시간표 값(-1.9h), 같은 고장 과거 평균(-0.3h), 악천후 x 표준시간(-0.1h)
 - R0255 fires/engine 예상 정비 10.8h(평균 대비 +6.7h), 영향 요인: 정비 표준시간표 값(+4.9h), 같은 고장 과거 평균(+1.1h), 핵심장비 x 부속 미확보(+1.0h)

경보: 22일차(훈련 주간) 부하율 1.0 초과 정비반 ['MT1', 'MT2', 'MT3', 'MT4', 'MT5', 'MT6'] -> 전 정비반 과부하 -> 상급 정비부대 지원 요청·비핵심 요청 연기 검토(정비대대장 판단)
```
</details>
